In [51]:
!pip install scikit-learn nltk pandas


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\SEC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [52]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import nltk

nltk.download('stopwords')
print("Vibe check: Imports successful!")

Vibe check: Imports successful!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SEC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [53]:
# The manual, perfectly balanced dataset (5 Positive, 5 Negative)
reviews = [
    # --- The Positive Vibes (1) ---
    "The plot of this sci-fi thriller was absolutely goated. 10/10.",
    "IPL is starting today and the energy at Chepauk is going to be insane!",
    "Massive game today, the squad played perfectly. Huge W.",
    "The visuals and cinematography were stunning, highly recommend.",
    "Everything is going perfectly, best phase of my life right now.",
    
    # --- The Negative Vibes (0) ---
    "This is the worst phase of my life, completely terrible.",
    "The plot was entirely predictable and the acting was pure trash.",
    "Worst match I have ever watched, absolutely boring and a waste of time.",
    "I would rather watch paint dry than see this garbage again.",
    "Terrible midfield performance, they completely threw the game away."
]

# The labels (1 = Positive, 0 = Negative)
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

print("Custom dataset locked in. 🚀")

Custom dataset locked in. 🚀


In [54]:
# 1. Setup the text-to-number converter (ignoring filler words like 'the', 'is')
vectorizer = TfidfVectorizer(stop_words='english')

# 2. Convert our text into a math matrix
X_features = vectorizer.fit_transform(reviews)

# 3. Create the ML model
model = MultinomialNB()

# 4. Train it on the data!
model.fit(X_features, labels)

print("Model is officially trained and ready to go!")

Model is officially trained and ready to go!


In [55]:
# Make up some new, unseen reviews
test_reviews = [
    "This thriller series had me on the edge of my seat the whole time! The plot twists were insane.",
    "Honestly, the finale of that sci-fi show was a massive letdown. Terrible writing."
]

# 1. Convert the new text using the EXACT SAME vectorizer from Step 4
test_features = vectorizer.transform(test_reviews)

# 2. Predict the sentiment
predictions = model.predict(test_features)

# 3. Print out the results cleanly
for review, pred in zip(test_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: '{review}'\nPredicted Vibe: {sentiment}\n")

Review: 'This thriller series had me on the edge of my seat the whole time! The plot twists were insane.'
Predicted Vibe: Positive

Review: 'Honestly, the finale of that sci-fi show was a massive letdown. Terrible writing.'
Predicted Vibe: Positive



In [56]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re

print("Phase 2 tools locked and loaded. 🚀")

Phase 2 tools locked and loaded. 🚀


In [57]:
# Pull 10,000 real Yelp reviews directly from the web
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/yelp.csv"
df = pd.read_csv(url)

# Keep only 1, 2, 4, and 5 star reviews
df = df[df.stars != 3]

# Create our labels: 1 for positive (4-5 stars), 0 for negative (1-2 stars)
df['sentiment'] = df['stars'].apply(lambda x: 1 if x > 3 else 0)

# Let's peek at the first 3 rows to make sure it worked
print(f"Total reviews locked in: {len(df)}")
df[['text', 'sentiment']].head(3)

Total reviews locked in: 8539


,text,sentiment
0,My wife took me here on my birthday for breakf...,1
1,I have no idea why some people give bad review...,1
2,love the gyro plate. Rice is so good and I als...,1


In [58]:
def clean_text(text):
    # Make everything lowercase
    text = text.lower()
    # Remove anything that isn't a letter (like punctuation or numbers)
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the cleaning function to all 8,000+ reviews
df['cleaned_text'] = df['text'].apply(clean_text)

print("Text data is squeaky clean. 🧼")

Text data is squeaky clean. 🧼


In [59]:
# Find out exactly how many negative reviews we have (the smaller group)
min_class_size = df['sentiment'].value_counts().min()

# Randomly sample the exact same amount of positive reviews
df_pos = df[df['sentiment'] == 1].sample(min_class_size, random_state=42)
df_neg = df[df['sentiment'] == 0].sample(min_class_size, random_state=42)

# Mash them together into a perfectly balanced dataset
df_balanced = pd.concat([df_pos, df_neg])

print(f"Dataset is now perfectly balanced! Total rows: {len(df_balanced)}")
# Let's check the balance:
print(df_balanced['sentiment'].value_counts())

Dataset is now perfectly balanced! Total rows: 3352
sentiment
1    1676
0    1676
Name: count, dtype: int64


In [60]:
# 1. Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42 # This just ensures the random split is exactly the same every time we run it
)

# 2. Setup our TF-IDF Vectorizer again
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# 3. Convert the TRAINING text into numbers and train the model
X_train_features = vectorizer.fit_transform(X_train)
model = MultinomialNB()
model.fit(X_train_features, y_train)

print("AI has finished studying the 80% training data! 🧠")

AI has finished studying the 80% training data! 🧠


In [61]:
# 1. Convert the TEST text into numbers (Notice we use .transform, NOT .fit_transform here!)
X_test_features = vectorizer.transform(X_test)

# 2. Ask the model to make predictions on the test exam
predictions = model.predict(X_test_features)

# 3. Calculate the accuracy
accuracy = accuracy_score(y_test, predictions)

print(f"🔥 Final Model Accuracy: {accuracy * 100:.2f}% 🔥")

🔥 Final Model Accuracy: 82.73% 🔥


In [62]:
def predict_vibe(custom_review):
    # 1. Clean the text using the function we built earlier
    cleaned = clean_text(custom_review)
    
    # 2. Convert to numbers (MUST use .transform, not .fit)
    features = vectorizer.transform([cleaned])
    
    # 3. Predict the sentiment
    prediction = model.predict(features)[0]
    
    # 4. Output a clean result
    if prediction == 1:
        return f"'{custom_review}' -> Positive 🟢"
    else:
        return f"'{custom_review}' -> Negative 🔴"

# Try typing your own wild reviews in here!
print(predict_vibe("The plot was completely predictable, but the visuals were nice."))
print(predict_vibe("I would rather watch paint dry than see this again."))
print(predict_vibe("Honestly? Goated movie. 10/10."))
print(predict_vibe("This is the worst phase of my life"))
print(predict_vibe("IPL is Starting today"))
print(predict_vibe("worst"))

'The plot was completely predictable, but the visuals were nice.' -> Positive 🟢
'I would rather watch paint dry than see this again.' -> Positive 🟢
'Honestly? Goated movie. 10/10.' -> Positive 🟢
'This is the worst phase of my life' -> Positive 🟢
'IPL is Starting today' -> Positive 🟢
'worst' -> Negative 🔴


In [63]:
!pip install gradio

  Using cached starlette-0.52.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached pydantic_core-2.41.5-cp311-cp311-win_amd64.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   - -------------------------------------- 2.1/43.0 MB 9.8 MB/s eta 0:00:05
   --- ------------------------------------ 3.9/43.0 MB 9.8 MB/s eta 0:00:04
   ------- -------------------------------- 7.6/43.0 MB 12.1 MB/s eta 0:00:03
   ---------- ----------------------------- 10.7/43.0 MB 12.7 MB/s eta 0:00:03
   ----------- ---------------------------- 12.1/43.0 MB 11.8 MB/s eta 0:00:03
   ------------ --------------------------- 13.4/43.0 MB 10.8 MB/s eta 0:00:03
   ------------- -------------------------- 14.7/43.0 MB 9.9 MB/s eta 0:00:03
   --------------- ------------------------ 16.8/43.0 MB 10.0 MB/s eta 0:00:03
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\SEC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [64]:
import gradio as gr

# 1. We tweak our function slightly just to return a clean string for the UI
def vibe_check_ui(custom_review):
    # Clean it up
    cleaned = clean_text(custom_review)
    
    # Convert to numbers (using the vectorizer we trained)
    features = vectorizer.transform([cleaned])
    
    # Predict
    prediction = model.predict(features)[0]
    
    # Return the vibe
    if prediction == 1:
        return "🟢 Positive Vibe Detected"
    else:
        return "🔴 Negative Vibe Detected"

# 2. Build the slick Gradio Interface
demo = gr.Interface(
    fn=vibe_check_ui,                    # The function we want it to run
    inputs=gr.Textbox(lines=3, placeholder="Type your movie review, IPL thoughts, or random rants here..."), # The input box
    outputs=gr.Text(label="AI Verdict"), # The output box
    title="Futzy's Vibe Checker AI 🧠",
    description="Drop a sentence below and my custom-trained Machine Learning model will catch the vibe.",
    theme="default"
)

# 3. Launch the server!
demo.launch()

C:\Users\SEC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv


In [65]:
import joblib

# 1. Save the actual trained AI brain
joblib.dump(model, 'sentiment_model.pkl')

# 2. Save the text-to-number translator
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Major W! 🏆 Model and Vectorizer have been secured to your hard drive.")

Major W! 🏆 Model and Vectorizer have been secured to your hard drive.
